In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.metrics import mean_squared_error

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DATA_PATH = "/content/drive/MyDrive/ML/Dataset/titanic/train.csv"
df =pd.read_csv(DATA_PATH)

X = df.drop("Survived", axis=1)
y= df["Survived"]

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# ubah format data gender
from sklearn.preprocessing import LabelEncoder
if df['Sex'].dtype == 'object':
    le = LabelEncoder()
    df['Sex'] = le.fit_transform(df['Sex'])

# Missing value di fare
import numpy as np
if df['Fare'].isnull().any():
    df['Fare'] = df['Fare'].fillna(df['Fare'].mean())

# Missing value di age
if df['Age'].isnull().any():
    df['Age'] = df['Age'].fillna(df['Age'].mean())

# Deteksi nilai Fare yang > 500 (tidak wajar untuk harga tiket)
misplaced_mask = pd.to_numeric(df["Fare"], errors="coerce") > 500

df.loc[misplaced_mask, "Ticket"] = df.loc[misplaced_mask, "Fare"]
df.loc[misplaced_mask, "Fare"] = np.nan

df["Fare"] = pd.to_numeric(df["Fare"], errors="coerce")

df["Ticket"] = df["Ticket"].astype(str).str.strip()
df["Ticket"] = df["Ticket"].replace("", np.nan)

print("\nData setelah preprocessing:")
print(df)

print("\nTipe data:")
print(df.dtypes)

print("\nJumlah nilai kosong:")
print(df.isnull().sum())


Data setelah preprocessing:
     PassengerId  Survived  Pclass  \
0              1         0       3   
1              2         1       1   
2              3         1       3   
3              4         1       1   
4              5         0       3   
..           ...       ...     ...   
886          887         0       2   
887          888         1       1   
888          889         0       3   
889          890         1       1   
890          891         0       3   

                                                  Name  Sex        Age  SibSp  \
0                              Braund, Mr. Owen Harris    1  22.000000      1   
1    Cumings, Mrs. John Bradley (Florence Briggs Th...    0  38.000000      1   
2                               Heikkinen, Miss. Laina    0  26.000000      0   
3         Futrelle, Mrs. Jacques Heath (Lily May Peel)    0  35.000000      1   
4                             Allen, Mr. William Henry    1  35.000000      0   
..                          

In [ ]:
from sklearn.linear_model import LogisticRegression

X = df.drop(columns=['PassengerId', 'Name', 'Ticket', 'Embarked', 'Cabin', 'Survived'], errors='ignore')
y = df['Survived']

cv_scores =np.sqrt(
    -cross_val_score(
        LinearRegression(),
        X,
        y,
        cv=5,
        scoring="neg_mean_squared_error"
    )
)

model = LogisticRegression(max_iter=200)

#akurasi
acc = cross_val_score(model, X, y, cv=5, scoring='accuracy')
print(f"Accuracy: {acc.mean():.3f} (+/- {acc.std():.3f})")

#f1 metrik
f1 = cross_val_score(model, X, y, cv=5, scoring='f1')
print(f"F1-score: {f1.mean():.3f} (+/- {f1.std():.3f})")

Accuracy: 0.781 (+/- 0.017)
F1-score: 0.707 (+/- 0.032)
